# 实战 29: PSM 倾向性得分匹配 - Lalonde 就业培训评估

## 📖 业务背景 (The Golden Standard)

**Lalonde Dataset** 是因果推断领域最著名的“圣经级”数据集。它源自美国 1970 年代的**国家就业支持示范项目 (National Supported Work, NSW)**。

- **核心问题**：参加就业培训 (`treat=1`) 到底能不能提高工人在 1978 年的实际收入 (`re78`)？
- **挑战**：参加培训的人往往本来就是处于弱势的群体（学历低、收入低）。如果我们直接对比“参加者”和“未参加者”的收入，可能会得出“培训反而降低了收入”的错误结论（选择性偏差）。
- **任务**：使用 **PSM (Propensity Score Matching)** 技术，消除这些混杂因素的影响，还原真实的培训效果。

---

## Step 1: 数据加载

我们将使用一份处理好的 Lalonde 数据集。

**字段说明**：
- `treat`: 是否参加培训 (1=Treatment, 0=Control) **[核心干预变量]**
- `re78`: 1978 年的实际收入 (Real Earnings in 1978) **[核心结果变量 Outcome]**
- `age`: 年龄
- `educ`: 受教育年限
- `race`: 种族 (black, hispanic)
- `married`: 是否已婚
- `nodegree`: 是否无学位
- `re74`, `re75`: 1974/1975 年的收入 (基线收入，重要的混杂因素)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 加载数据 (优先使用本地下载好的文件)
local_path = 'data/lalonde.csv'
url = "https://vincentarelbundock.github.io/Rdatasets/csv/Matching/lalonde.csv"

try:
    # 优先尝试本地加载
    df = pd.read_csv(local_path)
    # 有些版本的 lalonde 可能包含 Unnamed 索引列，去掉它
    if 'Unnamed: 0' in df.columns:
        df = df.drop(columns=['Unnamed: 0'])
    print(f"✅ 本地数据加载成功！({local_path})")
except FileNotFoundError:
    try:
        # 本地没有再尝试在线加载
        print(f"⚠️ 本地未找到文件，尝试在线下载...")
        df = pd.read_csv(url)
        if 'Unnamed: 0' in df.columns:
            df = df.drop(columns=['Unnamed: 0'])
        print(f"✅ 在线数据加载成功！")
        # 保存到本地以便下次使用
        import os
        os.makedirs('data', exist_ok=True)
        df.to_csv(local_path, index=False)
    except Exception as e:
        print(f"❌ 数据加载惨败：{e}")
        print("请检查网络或手动下载文件到 data/lalonde.csv")

# 简单预览
df.head()

## Step 2: 探索性数据分析 (EDA) & Naive Estimation

在做复杂的模型之前，先看看数据的原始状态。

### 💭 思考题 1
如果不做任何处理，直接计算 `Treatment组` 和 `Control组` 的收入差 (`re78` 均值之差)，结果是多少？这个结果可信吗？为什么？

In [ ]:
# 1. 查看两组的样本量
print("样本分布:\n", df['treat'].value_counts())

# 2. Naive Estimation (直接计算均值差)
naive_diff = df[df['treat']==1]['re78'].mean() - df[df['treat']==0]['re78'].mean()
print(f"\nNaive Effect (直接对比): ${naive_diff:.2f}")

# 3. 检查协变量是否平衡 (比如：参加培训的人是不是本身学历更低？)
print("\n基线特征对比 (均值):")
print(df.groupby('treat')[['age', 'educ', 'married', 'nodegree', 're74']].mean())

## Step 3: 倾向性得分估算 (Propensity Score Estimation)

我们需要计算每个样本“**参加培训的概率**” ($P(T=1|X)$)。
通常使用 Logistic 回归来拟合。

### 📝 任务
1.  定义特征 `X` (包括 age, educ, race, married, nodegree, re74, re75)。
2.  训练 Logistic Regression 模型预测 `treat`。
3.  将预测出的概率存为新的一列 `ps_score`。

In [ ]:
from sklearn.linear_model import LogisticRegression

# 1. 准备特征 (注意处理 categorical 变量，这里 lalonde 数据很多已经是 dummy 了)
# 如果 race 是字符串，需要 get_dummies。如果不确定，先 df.info() 看一下。
# 假设 race 已经是处理好的或者需要处理：

# TODO: 在这里写你的 Logistic 回归代码
# X = ...
# y = ...
# model = ...
# df['ps_score'] = model.predict_proba(X)[:, 1]

pass

## Step 4: 检查共同支撑域 (Common Support)

画出 Treatment 组和 Control 组的 `ps_score` 分布图。**重点观察是否有重叠部分**。
如果完全没重叠 (Perfect Separation)，说明无法匹配 (就是你上一个项目遇到的问题)。

In [ ]:
# TODO: 使用 sns.kdeplot 绘制两条曲线
pass

## Step 5: 进行匹配 (Matching)

最常用的方法是 **最近邻匹配 (Nearest Neighbor Matching)**。
我们可以使用 `sklearn.neighbors.NearestNeighbors` 来实现 1:1 匹配。

### 📝 任务
为每一个 Treated 样本，在 Control 组中找到一个 `ps_score` 最接近的样本。

In [ ]:
from sklearn.neighbors import NearestNeighbors

# 1. 分离两组数据
treated_df = df[df['treat'] == 1]
control_df = df[df['treat'] == 0]

# 2. 训练 KNN 模型 (在 Control 组上训练)
# TODO: 写你的匹配逻辑

# 3. 生成匹配后的 DataFrame (matched_df)
# matched_df 应该包含所有的 Treated 样本和它们匹配到的 Control 样本

pass

## Step 6: 平衡性检验 (Balance Check)

匹配完了，必须检查匹配质量。**匹配后的** Treatment 组和 Control 组，在各个协变量 (`age`, `educ`, `re74` 等) 上是否已经没有显著差异了？

通常我们看 **Standardized Mean Difference (SMD)**，或者直接对比均值。

In [ ]:
# TODO: 对比 matched_df 中 Treat 和 Control 组的特征均值
pass

## Step 7: 估算 ATT (Average Treatment Effect on the Treated)

现在数据平衡了，我们可以直接计算均值差了！

$$ ATT = E[Y | T=1, Matched] - E[Y | T=0, Matched] $$

In [ ]:
# TODO: 计算匹配后的 re78 差值
pass

## 🏁 结论

你的 PSM 估算结果是多少？
相比于最开始的 Naive Estimation，结果发生了什么变化？说明了什么？

> **💡 参考答案**：Lalonde 实验的真实基准结果 (Benchmark) 大约是 **$1794** (正向收益)。
> 看你的 PSM 结果是否比 Naive 结果更接近这个数字？